# FoF - Fear of Forgetting

My nostalgia has always led me to fear losing track and control over my memories, hence I've been trying to capture all my memories since high school through pictures and diary pages. I noticed that during uni my habits in this regard changed, with more things to do and less time to take pictures but i focused on writing more.

This study investigates the evolution of my **Fear of Forgetting habits**.

The first step consists in renaming the columns of my dataset, from natural language to snake case.

In [1]:
import pandas as pd

df = pd.read_csv('fof_dataset.csv')

rename_rules = {
    'university year': 'university_year',
    'iphone photos and videos': 'iphone_media_count',
    'camera photos and videos': 'camera_media_count',
    'journal pages': 'journal_pages',
    'my age': 'my_age',
    'academic period': 'academic_period',
    'exams prepared': 'exams_prepared',
    'hours of work': 'work_hours',
    'travel days': 'travel_days'
}

df = df.rename(columns=rename_rules)

df.head()

,month,iphone_media_count,camera_media_count,journal_pages,my_age,university_year,academic_period,exams_prepared,work_hours,travel_days
0,Sep-21,667,270,2,18,1,lectures,0,0,0
1,Oct-21,753,884,3,18,1,lectures,0,0,0
2,Nov-21,546,589,10,18,1,lectures,0,0,0
3,Dec-21,713,1181,5,19,1,lectures,1,0,0
4,Jan-22,558,507,2,19,1,exam session,1,0,0


## Did I actually stop taking as many pictures as the years went on?

In this next step I visualise if the number of pictures taken on my phone and camera decreased, or if it is only a feeling of mine.

In [21]:
import plotly.express as px
import pandas as pd

# make month string into date
df['date_parsed'] = pd.to_datetime(df['month'], format='%b-%y')
df = df.sort_values('date_parsed')

# create line graph
fig = px.line(
    df, 
    x='date_parsed', 
    y=['camera_media_count', 'iphone_media_count'],
    color_discrete_sequence=['#355c7d', '#c06c84'], 
    title='<b>The Digital Deceleration: Evolution of Photo Captures</b>',
    labels={'date_parsed': 'Timeline', 'value': 'Media Captured'}
)

legend_names = {
    'camera_media_count': 'Camera Photos & Videos', 
    'iphone_media_count': 'iPhone Photos & Videos'
}
fig.for_each_trace(lambda trace: trace.update(name = legend_names[trace.name]))

# make tooltip
for trace in fig.data:
    trace.hovertemplate = f"<b>{trace.name}</b>: %{{y:,}}<extra></extra>"
    trace.line.width = 3

# line graph layout
fig.update_layout(
    font_family="Times New Roman",
    title_font_size=18,
    title_x=0.5,
    xaxis_title="University Years Timeline",
    yaxis_title="Monthly Media Captured",
    plot_bgcolor="white",
    width=1000,
    height=550,
    hovermode="x unified", 
    legend=dict(
        yanchor="top", 
        y=0.98, 
        xanchor="right", 
        x=0.98, 
        bgcolor="rgba(255,255,255,0.7)", 
        title_text=""
    )
)

fig.update_xaxes(showgrid=True, gridcolor='#EAEAEA', tickformat='%b-%y', dtick="M3", tickangle=45)
fig.update_yaxes(showgrid=True, gridcolor='#EAEAEA')

fig.show()

This graph shows how my idea was technically correct, as the number of total media captured across the two devices has indeed decreased with time. 
I notice that the camera count decreases more compared to the iphone one, because I carry my camera around mich less, as most days only consist in studying and not doing anything extra.  
It seems generalluy that the dicrease started in 2025.

## What changed after 2025?

In this section I visualize the increase in busy-ness in the last two years, which I personally think is the cause for the dicrease of media captures.

In [47]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd



# create 2 stacked graphs sharing x axis 
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.08, 
    subplot_titles=("<b>Exams Prepared per Month</b>", "<b>Hours of Work per Month</b>")
)

# upper graph
fig.add_trace(
    go.Scatter(
        x=df['date_parsed'], y=df['exams_prepared'], 
        name="Exams Prepared", mode='lines+markers',
        line=dict(color='#355c7d', width=3.5),
        marker=dict(size=5),
        showlegend=True
    ),
    row=1, col=1
)

# bottom graph
fig.add_trace(
    go.Scatter(
        x=df['date_parsed'], y=df['work_hours'], 
        name="Hours of Work", mode='lines+markers',
        line=dict(color='#c06c84', width=3),
        marker=dict(size=5, symbol='diamond'),
        showlegend=True
    ),
    row=2, col=1
)

# data viz palette
period_colors = {
    'lectures': 'rgba(124, 143, 161, 0.18)',         
    'exam session': 'rgba(169, 135, 199, 0.18)',     
    'summer break': 'rgba(201, 143, 93, 0.18)',      
    'bachelor graduation': 'rgba(170, 92, 92, 0.18)'  
}

legend_marker_colors = {
    'lectures': '#7C8FA1',
    'exam session': '#A987C7',
    'summer break': '#C98F5D',
    'bachelor graduation': '#AA5C5C'
}

# legend and regions for academic periods
for period in period_colors.keys():
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None], 
            mode='markers',
            marker=dict(color=legend_marker_colors[period], size=14, symbol='square'),
            name=f"Period: {period.title()}",
            showlegend=True
        )
    )

for i in range(len(df)):
    row_data = df.iloc[i]
    x0 = row_data['date_parsed']
    x1 = df.iloc[i+1]['date_parsed'] if i < len(df) - 1 else x0 + pd.DateOffset(months=1)
    
    fig.add_vrect(
        x0=x0, x1=x1,
        fillcolor=period_colors[row_data['academic_period']],
        layer="below", line_width=0
    )

# graph layout
fig.update_layout(
    font_family="Times New Roman",
    title_text="<b>The Compounding Burden: Five Years of Escalating Academic and Work Demands</b>",
    title_font_size=22,
    title_x=0.5,
    plot_bgcolor="white",
    width=1100, 
    height=650,
    hovermode="x unified",
    legend=dict(
        orientation="v",       
        yanchor="top",         
        y=1,
        xanchor="left",        
        x=1.03,                
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#E8E8E8",
        borderwidth=1
    )
)

fig.update_xaxes(showgrid=True, gridcolor='#F3F3F3', tickformat='%b-%y', dtick="M3", tickangle=45, range=['2021-08-01', '2026-05-01'], row=2, col=1, title_text="University Years Timeline")
fig.update_xaxes(showgrid=True, gridcolor='#F3F3F3', range=['2021-08-01', '2026-05-01'], row=1, col=1)
fig.update_yaxes(showgrid=True, gridcolor='#F3F3F3', title_text="Exam Count", row=1, col=1)
fig.update_yaxes(showgrid=True, gridcolor='#F3F3F3', title_text="Work Hours", row=2, col=1)

fig.show()